# Modelo realista · sin datos del futuro (Pablo · `realista/`)

> **Separado de `futuro/` a propósito.** Aquí el modelo solo ve el **snapshot actual y snapshots PASADOS** de la sesión,
> como funcionaría en tiempo real. Sin snapshots posteriores y sin el oráculo (etiquetas de otros controladores).
> No es `main` (la regla 1 solo permite el snapshot) ni `futuro/`. Prueba por curiosidad (21/10): ¿cuánto se puede
> mejorar sin mirar al futuro?

**Código** (reutiliza por importación funciones de `futuro/` sin modificarlas):
- `realista/features_real.py`: columnas que solo miran atrás; un `assert` falla si se cuela una que mira al futuro
  (`blk_len`, `gap_next`, `gxc_exitfl_next`).
- `realista/feat_causal.py`: bloque `causal` (`cz_*`): segundos desde el último cambio del estado que resuelve cada tarea
  (coordinación de entrada/salida, assume, nivel autorizado, rumbo, nivel de salida, zona, conflicto, ruta).
- `realista/feat_session_causal.py`: segundo paso de sesión que solo usa scores de snapshots anteriores: media en la
  racha hasta ahora, 1 y 2 snapshots antes, máximo en rachas anteriores («ya se hizo»), racha anterior y avión.
  Comprobado sin fuga: al alterar las puntuaciones futuras de una sesión no cambia ninguna fila anterior (0 de 6034).
- `realista/m_lgb_real.py`: mismos parámetros, folds y semillas que el LightGBM de futuro. Guarda en
  `realista/oof|preds|submissions` y apunta en `experiments/lgb_real_runs.csv`.

**Bloques:** `base` = columnas originales (las `a_*`, `x_*`, `k_*` del reto ya miran solo atrás) · `logic` = reglas
dentro del snapshot · `past` = `p_*`, `blk_pos`, `gap_prev` · `causal` = `cz_*` · `geo` = velocidad de acercamiento al
borde, tráfico que cruza el mismo borde y niveles (de `futuro/feat_geo.py`, sin lo que mira al futuro).

## 1 · Resultados (3 semillas, `Data/folds.csv`)

Referencia: solo el snapshot (`real_base`) = **0.6572**, el mismo nivel que el LightGBM de `main` (0.655).

| Modelo | Bloques | 1.er pase | + segundo paso causal | peor controlador (+ sesión) | frente a `real_base_ses`: sesiones / controladores suben–bajan |
|---|---|---|---|---|---|
| `real_base` | base | 0.6572 | 0.6556 | 0.568 (016) | — |
| `real_blp` | + logic + past | 0.6571 | 0.6597 | 0.591 (016) | 20/17 · 10/8 |
| `real_blpc` | + causal | 0.6645 | 0.6617 | 0.590 (016) | 26/11 · 11/6 |
| **`real_todo`** | **+ geo** | **0.6703** | **0.6718** | **0.600 (016)** | **28/9 · 15/3** |

- **Mejor modelo realista: `real_todo_ses` = 0.6718**, +1.5 sobre solo el snapshot. Mediana por sesión 0.684 (+1.2
  frente a `real_base_ses`); 15 de 18 controladores suben.
- **Lo que aporta es el pasado del avión:** cuánto hace que cambió el estado ligado a la tarea (`causal`, +0.7) y su
  movimiento reciente y el tráfico en el borde (`geo`, +0.6). `logic` y `past` no aportan.
- **El segundo paso causal apenas ayuda** (−0.3 a +0.3). Sin ver qué pasa después, el «explaining away» de la
  sesión pierde casi toda su fuerza: en futuro daba +0.7.
- **Comparación:** con futuro, el mismo LightGBM llega a 0.914 con sesión. El salto de ~0.67 a ~0.91 lo da casi entero
  saber qué hace el controlador después (siguiente instrucción, cuándo desaparece la tarea).

Envío posible (no subido): `realista/submissions/sub_real_todo_ses.csv`.


In [ ]:
# Reproducir (terminal, desde la raíz; ~8 min cada una, se pueden lanzar a la vez con --threads 7):
#   python realista/m_lgb_real.py --sesion --name real_base --feats base
#   python realista/m_lgb_real.py --sesion --name real_todo --feats base,logic,past,causal,geo --ref real_base
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "realista" else Path.cwd()
sys.path[:0] = [str(ROOT / "futuro"), str(ROOT / "cv")]
import pandas as pd, metricas
for n in ["real_base", "real_blp", "real_blpc", "real_todo"]:
    for s in ("", "_ses"):
        o = pd.read_csv(ROOT / "realista" / "oof" / f"{n}{s}.csv")
        r = pd.read_csv(ROOT / "realista" / "oof" / f"real_base{s}.csv")
        print(metricas.resumen(n + s, metricas.metricas(o, r), "real_base" + s))